# Unit 5: Finance - Classical Pricing and Toy Amplitude Estimation

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** toy demonstration


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook splits the finance story in two:

1. price a European call option with Black-Scholes and classical Monte Carlo;
2. run a compiled toy amplitude-estimation phase readout for a discretised exercise-probability proxy.

It does **not** encode the full continuous payoff into amplitudes, and it does **not** claim to price the option quantumly on Quokka.

**What you'll see:**
1. Compute the Black-Scholes analytical price
2. Classical Monte Carlo and its $1/\sqrt{N}$ convergence
3. Discretise terminal prices into 8 bins and mark the in-the-money bins
4. Compile the corresponding Grover eigenphase for that binary exercise event
5. Read back that phase on Quokka and recover the exercise fraction

In [ ]:
import json

import numpy as np

import matplotlib.pyplot as plt

from scipy.stats import norm



import requests

from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)



QUOKKA = "quokka1"

QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"



def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(
        QUOKKA_URL,
        json={"script": program, "count": shots},
        verify=False,
        timeout=30,
    )
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. The option and its analytical price

European call option: right to buy a stock at price $K$ (strike) at time $T$ (maturity).

Payoff: $\max(S_T - K, 0)$

In [ ]:
# Option parameters
S0 = 100     # current stock price
K = 105      # strike price
T = 1.0      # maturity (years)
r = 0.05     # risk-free rate
sigma = 0.20 # volatility

# Black-Scholes analytical price
d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
bs_price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

print(f"Black-Scholes price: ${bs_price:.4f}")

## 2. Classical Monte Carlo

In [ ]:
# Monte Carlo convergence
np.random.seed(42)
sample_sizes = [10, 50, 100, 500, 1000, 5000, 10000, 50000]
mc_prices = []
mc_errors = []

for N in sample_sizes:
    Z = np.random.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0) * np.exp(-r * T)
    mc_price = np.mean(payoffs)
    mc_err = np.std(payoffs) / np.sqrt(N)
    mc_prices.append(mc_price)
    mc_errors.append(mc_err)
    print(f"N={N:>6}: price=${mc_price:.4f} ± ${mc_err:.4f}")

print(f"\nTrue price: ${bs_price:.4f}")

In [ ]:
# Plot convergence: error vs sample size
plt.figure(figsize=(8, 5))
plt.loglog(sample_sizes, mc_errors, 'o-', color='#FF5722', label='Classical MC error')

# 1/sqrt(N) reference line
N_ref = np.array(sample_sizes)
plt.loglog(N_ref, mc_errors[0] * np.sqrt(sample_sizes[0]) / np.sqrt(N_ref),
           '--', color='gray', alpha=0.5, label='$1/\sqrt{N}$ reference')

# Hypothetical quantum 1/N convergence
plt.loglog(N_ref, mc_errors[0] * sample_sizes[0] / N_ref,
           '--', color='#009688', alpha=0.7, label='Quantum $1/N$ (projected)')

plt.xlabel('Number of samples / queries')
plt.ylabel('Estimation error ($)')
plt.title('Monte Carlo convergence: classical vs. quantum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Toy amplitude estimation for a discretised exercise probability

The full pricing algorithm would encode both the log-normal distribution and the normalised payoff into amplitudes.

This notebook does something narrower and honest: it keeps the price calculation classical, reduces the quantum part to a binary question (is the option in the money or not?), and then runs a compiled amplitude-estimation phase-readout toy for that discretised exercise probability.

In [ ]:
# Discretise stock price into 8 bins and mark the in-the-money bins
n_qubits = 3
n_bins = 2 ** n_qubits

price_range = np.linspace(S0 * 0.5, S0 * 1.5, n_bins + 1)
bin_centers = 0.5 * (price_range[:-1] + price_range[1:])
in_the_money = bin_centers > K
exercise_fraction = float(np.mean(in_the_money))
n_itm = int(np.sum(in_the_money))

print('Bin | Price range      | In the money?')
print('-' * 43)
for i, (lo, hi) in enumerate(zip(price_range[:-1], price_range[1:])):
    label = 'yes' if in_the_money[i] else 'no'
    print(f" {i:>2} | ${lo:6.1f} - ${hi:6.1f} | {label}")

print()
print(f"{n_itm}/{n_bins} bins in the money")
print(f"Exercise fraction on this grid: {exercise_fraction:.3f}")

In [ ]:
# Compiled toy QAE phase readout for the discretised exercise fraction
ancilla_bits = 3
levels = 2 ** ancilla_bits

theta_exact = float(np.arcsin(np.sqrt(exercise_fraction)))
phase_exact = theta_exact / np.pi
phase_integer = int(np.clip(np.rint(phase_exact * levels), 0, levels - 1))
encoded_phase = phase_integer / levels
expected_peak = format(phase_integer, f"0{ancilla_bits}b")
phase_angle = 2 * np.pi * encoded_phase
encoded_exercise_fraction = float(np.sin(np.pi * encoded_phase) ** 2)

print(f"Exact exercise fraction on the 8-bin grid: {exercise_fraction:.4f}")
print(f"Grover angle theta:                     {theta_exact:.4f} rad")
print(f"Exact phase theta/pi:                  {phase_exact:.4f}")
print(f"Encoded 3-bit phase fraction:          {encoded_phase:.4f} ({expected_peak})")
print(f"Exercise fraction on that 3-bit grid:  {encoded_exercise_fraction:.4f}")

qae_toy_circuit = f"""OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];  // q[0-2] = ancilla register, q[3] = eigenstate register
creg c[3];

// Prepare the one-qubit eigenstate |1>
x q[3];

// Put the ancilla register into superposition
h q[0];
h q[1];
h q[2];

// Compiled controlled-G^(2^k) phases for the encoded Grover eigenphase
cu1({phase_angle:.6f}) q[2], q[3];
cu1({2 * phase_angle:.6f}) q[1], q[3];
cu1({4 * phase_angle:.6f}) q[0], q[3];

// Inverse QFT on the ancilla register
h q[0];
cu1({-np.pi / 2:.6f}) q[0], q[1];
h q[1];
cu1({-np.pi / 4:.6f}) q[0], q[2];
cu1({-np.pi / 2:.6f}) q[1], q[2];
h q[2];

// Bit reversal swap
cx q[0], q[2];
cx q[2], q[0];
cx q[0], q[2];

measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

results = run_qasm(qae_toy_circuit, shots=1024)

print()
print('Compiled toy QAE measurement results:')
for outcome in sorted(results.keys(), key=lambda bitstring: results[bitstring], reverse=True):
    phase_fraction = int(outcome, 2) / levels
    exercise_proxy = float(np.sin(np.pi * phase_fraction) ** 2)
    print(f"  {outcome}: {results[outcome]:>4} counts -> phase {phase_fraction:.3f} -> exercise fraction {exercise_proxy:.3f}")

best_outcome = max(results, key=results.get)
peak_probability = results[best_outcome] / sum(results.values())
phase_estimate = int(best_outcome, 2) / levels
exercise_prob_qae_toy = float(np.sin(np.pi * phase_estimate) ** 2)
phase_grid_error = abs(exercise_prob_qae_toy - exercise_fraction)

print()
print(f"Dominant outcome:                {best_outcome} (expected {expected_peak})")
print(f"Recovered phase fraction:        {phase_estimate:.4f}")
print(f"Recovered exercise fraction:     {exercise_prob_qae_toy:.4f}")
print(f"Exact exercise fraction:         {exercise_fraction:.4f}")
print(f"Grid approximation error:        {phase_grid_error:.4f}")
print(f"Peak probability:                {peak_probability:.3f}")
print('This demonstrates compiled amplitude-estimation phase readout, not full quantum option pricing.')

## What's next?

- [Deep-Dive 5 - Amplitude Estimation from Grover](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/10-amplitude-estimation.md): the algorithmic machinery behind the phase readout
- Replace the binary exercise flag with ancilla payoff rotations to move from this toy to real payoff encoding
- Swap the compiled Grover eigenphase for an explicitly constructed Grover iterator when the circuit budget allows it
- [Unit 6 - Supply Chains](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/11-supply-chains.md): where optimisation, not sampling, becomes the bottleneck